# The test dataset:

https://demo.borealisdata.ca/dataset.xhtml?persistentId=doi:10.80240/FK2/WEYHSP

# Change to the Juypter Server directory

In [1]:
cd ~/MDL/pydatacuration

/home/jovyan/MDL/pydatacuration


# Import the libraries

In [2]:
import os
import sys
import yaml
import pandas as pd
import datetime
import janitor
import hashlib
import subprocess
import pydatacuration.utils as utils
import time

In [3]:
# Load the configuration file
with open('config.yaml', 'r') as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

In [4]:
# Create log files directory
utils.mk_log_dir()

'./log_files'

# Output

In [5]:
# Redirect Tree structure to a text file
original_stdout = sys.stdout # Save a reference to the original standard output
with open('./log_files/ds_structure.txt', 'w') as f:
    sys.stdout = f # Change the standard output to the file we created.
    utils.list_files('data')
    sys.stdout = original_stdout # Reset the standard output to its original value

In [6]:
# Export the structure ('tree') of a directory and its files as a dictionary
def get_filepaths(directory):
    file_info = {}
    id = 1  # Initialize id outside the loop
    # Walk the tree
    for root, dir, files in os.walk(directory):
        for file in files:
            full_file_path = os.path.join(root, file)
            parent_directory = os.path.basename(os.path.dirname(full_file_path))
            file_info[id] = {  # Use id as a key in file_info
                'root': directory,
                'parent_directory': parent_directory,
                'depth': full_file_path.count(os.sep) - directory.count(os.sep) + 1,
                'file': file,
                'file_path': full_file_path
            }
            id += 1  # Increment id after adding the file info to the dictionary
    return file_info

In [7]:
def sha256sum(filepath):
    with open(filepath, 'rb', buffering=0) as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()

In [8]:
df = pd.DataFrame.from_dict(get_filepaths('data'), orient='index')

In [9]:
# Update existing DataFrame in place
for index, row in df.iterrows():
    file_path = row['file_path']
    if os.path.exists(file_path):
        created = datetime.datetime.fromtimestamp(os.path.getctime(file_path))
        modified = datetime.datetime.fromtimestamp(os.path.getmtime(file_path))
        file_extension = os.path.splitext(file_path)[1]
        sha256_hash = sha256sum(file_path)
        df.at[index, 'file_id'] = index
        df.at[index, 'created'] = created
        df.at[index, 'modified'] = modified
        df.at[index, 'file_extension'] = file_extension
        df.at[index, 'sha256_hash'] = sha256_hash
    else:
        df.at[index, 'created'] = None
        df.at[index, 'modified'] = None

# Create a new DataFrame by copying the updated one
new_df = df.copy()
new_df = df.reorder_columns(['file_id', 'root'])

# Convert file_id to integer
new_df['file_id'] = new_df['file_id'].astype(int)

In [10]:
new_df.to_csv('./log_files/ds_file_info.csv', index=False)

# OPF-Fido implementation

In [11]:
subprocess.Popen([f'cd ~/MDL/pydatacuration/ | fido -recurse -zip data/ > log_files/temp_data/fileFormats_temp.csv'], shell=True)
time.sleep(2)

FIDO v1.6.1 (formats-v109.xml, container-signature-20200121.xml, format_extensions.xml)
FIDO: Zero byte file (empty): Path is: data/root_text.txt
FIDO: Processed     11 files in 310.68 msec, 35 files/sec


In [12]:
file_formats_df = pd.read_csv('log_files/temp_data/fileFormats_temp.csv')
os.remove('log_files/temp_data/fileFormats_temp.csv')
os.rmdir('log_files/temp_data')
file_formats_df.columns = ['fido.status', 'info.time', 'info.puid', 'info.formatname', 'info.signaturename', 'info.filesize', 'info.filename', 'info.mimetype', 'info.matchtype'] 

In [13]:
# Prepare the DataFrames (if necessary)
new_df['file_path'] = new_df['file_path'].str.strip()
file_formats_df['info.filename'] = file_formats_df['info.filename'].str.strip()

# Perform a left join
dataCuration_df = pd.merge(new_df, file_formats_df, left_on='file_path', right_on='info.filename', how='left')

In [14]:
dataCuration_df.drop(columns=['info.filename'], inplace=True)
dataCuration_df.to_csv('./log_files/dataCuration.csv', index=False)